# Operating envelopeWhat lateral acceleration the vehicle can physically reach, and on which surfacesthat is enough to saturate the tires. This decides the radii and speeds used insteps 5.7 and 5.8: if the platform cannot reach the friction limit, the tiresaturation parameters are not observable and no amount of fitting recovers them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

L = 0.173          # wheelbase [m]
T = 0.1745         # track [m]
R_WHEEL = 0.03415  # [m], stock tires
G = 9.81

DELTA_MAX = np.deg2rad(30.0)   # nominal, validated in step 5.5
V_MAX = 1.37                   # measured on the ground, loaded (step 5.1)
H_COG = 0.055                  # estimate, measured in step 5.3

## Max speedMeasured by ramping the `/cmd_vel` setpoint well past the achievable speed andwatching the wheels plateau. `VEHICLE_MAX_LINEAR_VEL_MPS` only clamps the serialtelemetry path, so the PID saw the full setpoint and saturated.| setpoint [m/s] | plateau [rad/s] ||---|---|| 1.7 | 39.1 / 40.1 || 2.2 | same || 2.6 | same |Both wheels sit at their own free speed and differ by 2.6%, which is whatsaturation looks like: with the loop in control they would track the samesetpoint. Raised the vehicle reaches 41.1 rad/s, so load costs 3.5% of freespeed, i.e. the motors deliver ~3.6% of stall torque at top speed. The limit isfree speed (voltage and gearing), not torque, and there is no torque reserveleft at v_max.

## Kinematic turn radiusAt low slip the bicycle model gives$$R = \frac{L}{\tan \delta}$$so full lock sets the tightest circle the vehicle can draw.

In [ ]:
R_min = L / np.tan(DELTA_MAX)
print(f"R_min = {R_min:.3f} m")

## EnvelopeOn a circle of radius $R$ at speed $v$ the lateral acceleration is $a_y = v^2/R$.With $v \le v_{max}$ and $R \ge R_{min}$:$$a_y^{max}(R) = \frac{v_{max}^2}{R}, \qquad R \ge R_{min}$$The tire saturates where this exceeds the friction limit $\mu g$.

In [ ]:
R = np.linspace(R_min, 2.0, 400)
ay_env = V_MAX**2 / R

mus = [0.15, 0.25, 0.35, 0.60]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(R, ay_env, 'k', lw=2, label=f'envelope, v_max = {V_MAX} m/s')
for mu in mus:
    ax.axhline(mu * G, ls='--', lw=1, label=f'mu = {mu}')
ax.axvline(R_min, color='r', lw=1, label=f'R_min = {R_min:.2f} m')
ax.set_xlabel('R [m]')
ax.set_ylabel('a_y [m/s^2]')
ax.set_ylim(0, 8)
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
mu_max = V_MAX**2 / (R_min * G)
print(f"mu_max reachable (kinematic) = {mu_max:.2f}")

## Operating pointFor a given $\mu$, a steady circle at the limit needs $v = \sqrt{\mu g R}$. Largerradii need more speed, but the locked rear axle scrubs less: the outer wheel wouldtravel$$\frac{R + T/2}{R - T/2} - 1$$more than the inner one, and cannot. Both wheels burn part of their frictionbudget longitudinally, leaving less for cornering, and the amount depends on $R$.Scrub is the column that decides: the largest radius with speed margin wins, andthat needs the lowest $\mu$.

In [ ]:
rows = []
for mu in mus:
    for Rc in [0.35, 0.5, 0.7, 0.9, 1.2]:
        if Rc < R_min:
            continue
        v_need = np.sqrt(mu * G * Rc)
        rows.append({
            'mu': mu,
            'R [m]': Rc,
            'v needed [m/s]': round(v_need, 2),
            'v / v_max': round(v_need / V_MAX, 2),
            'spool scrub': round((Rc + T/2) / (Rc - T/2) - 1, 2),
            'space [m]': round(2 * Rc + 0.6, 1),
        })

df = pd.DataFrame(rows)
df['feasible'] = df['v / v_max'] < 0.95
df

## Rollover checkThe vehicle tips instead of sliding above$$a_y = \frac{T/2}{h_{cog}} g$$which for a chassis this low sits far beyond the friction limit.

In [ ]:
ay_rollover = (T / 2) / H_COG * G
print(f"rollover threshold = {ay_rollover:.1f} m/s^2 = {ay_rollover/G:.2f} g")
print(f"envelope max       = {V_MAX**2/R_min:.1f} m/s^2 = {V_MAX**2/(R_min*G):.2f} g")

## Circle test, stock tires on tilesConstant radius, five speeds. `/cmd_vel` was scaled to hold $\omega/v$ constant,so the firmware commanded the same $\delta$ throughout. Yaw rate from the gyro.Note on the IMU: `imu_link` is rotated -90 deg about Z, so$a_y^{veh} = -a_x^{imu}$ and the raw `linear_acceleration.y` is longitudinal.The gyro is unaffected: a rotation about Z leaves the Z axis alone. Lateralacceleration here is reconstructed as $a_y = v \cdot r$ instead.

In [ ]:
v_cmd = np.array([0.4, 0.6, 0.8, 1.0, 1.2])
w_cmd = np.array([1.14, 1.71, 2.29, 2.86, 3.43])
r_meas = np.array([0.83, 1.22, 1.61, 2.02, 2.32])   # gyro mean at steady state

delta = np.arctan(w_cmd * L / v_cmd)
R_cmd = L / np.tan(delta)
R_act = v_cmd / r_meas
ay_act = v_cmd * r_meas

t = pd.DataFrame({
    'v cmd': v_cmd,
    'w cmd': w_cmd,
    'r meas': r_meas,
    'r / w': np.round(r_meas / w_cmd, 3),
    'R act [m]': np.round(R_act, 3),
    'ay [m/s^2]': np.round(ay_act, 2),
})
print(f"delta = {np.rad2deg(delta[0]):.1f} deg, R_cmd = {R_cmd[0]:.2f} m")
t

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ay_act, r_meas / w_cmd, 'o-')
ax.set_xlabel('a_y [m/s^2]')
ax.set_ylabel('r measured / r commanded')
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
plt.show()

print(f"ay reached = {ay_act.max():.2f} m/s^2 = {ay_act.max()/G:.2f} g")
print(f"R inflation = {np.mean(R_act/R_cmd):.2f}x")

The ratio holds flat near 0.71 across a 3x range of speed. A saturating tirewould drop it: the car would run wide and yaw rate would fall behind. It doesnot, so **stock tires on tiles never reach the limit**, and $\mu > 0.64$.The constant 29% shortfall is not friction. It is a fixed error, present at everyspeed, from the steering ratio, spool scrub (66% at this radius) and wheel slip.Step 5.5 separates the steering ratio from the rest.### The envelope is optimisticThe measured radius is 37% larger than the kinematic one, and $a_y = v^2/R$ has$R$ in the denominator. Redoing the ceiling with the real radius:

In [ ]:
infl = np.mean(R_act / R_cmd)
R_min_real = R_min * infl
mu_max_real = V_MAX**2 / (R_min_real * G)
print(f"R_min real  = {R_min_real:.2f} m")
print(f"mu_max real = {mu_max_real:.2f}")

And inflation grows towards the limit, so the true ceiling is lower still.Target for the slicks: **mu <= 0.2**.

## Surfaces| surface | tires | mu | method | saturable ||---|---|---|---|---|| tiles | stock | > 0.64 | circle test, r/w flat at 0.71 up to 0.28 g | no |## Test planPending slick tires.